# Describe the emitted reference

The output schema names the property that a consumer will receive.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from jsonschema import Draft202012Validator
from pydantic import BaseModel

from aoa.action_machine.domain import BaseEnumEntity, value


class CurrencyEnumEntity(BaseEnumEntity):
    """Describe the supported currency."""
    symbol: str
    EUR = value(code="eur", display_name="Euro", symbol="€")

class Price(BaseModel):
    """Hold a dictionary reference."""
    currency: CurrencyEnumEntity

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
schema = Price.model_json_schema(mode="serialization")
reference_schema = schema["properties"]["currency"]
price = Price(currency=CurrencyEnumEntity.EUR)
Draft202012Validator(schema).validate(price.model_dump(mode="json"))
properties = reference_schema["properties"]["properties"]
print("Required output:", reference_schema["required"])
print("Property definitions:", properties["properties"])
print("Extra keys:", properties["additionalProperties"])
assert properties["properties"]["symbol"] == {"type": "string"}

Required output: ['code', 'display_name', 'properties']
Property definitions: {'symbol': {'type': 'string'}}
Extra keys: False


## What the result means

The output has code, display_name and properties. The nested properties schema describes symbol as a string and rejects undeclared keys. In the current implementation the nested keys themselves are not listed as required: this schema is not a guarantee that every property key is present in an arbitrary document.